[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/03_datasets_experiments.ipynb)

# Lab 3: Iterate with Datasets & Experiments
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~30 min

**In this lab you will:**
- Turn **failing production traces into a dataset**
- Try a prompt fix in **Prompt Playground**
- Run **experiments** on a golden dataset: baseline (v1) vs. improved prompt (v2) vs. a cheaper model
- **Compare results side by side** to confirm the change actually helped

In [ ]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

# Model provider for the agent and the judges: "openai", "anthropic", "azure", or "bedrock".
# Leave blank to use LLM_PROVIDER from .env / Colab Secrets (default: openai).
LLM_PROVIDER = ""
PROJECT = load_credentials(provider=LLM_PROVIDER)  # prompts for keys if they aren't in .env / Colab Secrets

## 1. Traces → dataset

Start with production failures. We take traces flagged in Lab 2 and save them as a dataset. That dataset becomes a regression suite every future prompt has to pass.

> You can do this in the UI too: filter the traces table → select rows → **Add to Dataset**.

In [ ]:
from datetime import datetime
import pandas as pd
from workshop_utils import arize_client, export_spans, root_spans

client = arize_client()
SPACE_ID = os.environ["ARIZE_SPACE_ID"]
handle = PROJECT.removeprefix("civicassist-")
stamp = datetime.now().strftime("%H%M")

traces = root_spans(export_spans(hours=24))
label_cols = [c for c in traces.columns if c.startswith("eval.") and c.endswith(".label")]
print("Eval columns on traces:", label_cols or "(none yet: did you finish Lab 2?)")

def col(name):
    return traces[name] if name in traces else pd.Series("", index=traces.index)

if label_cols:
    bad = traces[col("eval.hallucination.label").eq("hallucinated") | col("eval.tool_call_accuracy.label").eq("incorrect")]
else:
    bad = traces.head(10)  # fallback so you can keep going

failures_df = bad[["input", "output"]].rename(columns={"input": "question", "output": "v1_answer"}).dropna()
print(f"{len(failures_df)} failing traces → dataset")
failures_df.head()

In [ ]:
if len(failures_df):
    failures_ds = client.datasets.create(
        name=f"civicassist-prod-failures-{handle}-{stamp}",
        space=SPACE_ID,
        examples=failures_df.to_dict(orient="records"),
    )
    print("Created dataset:", failures_ds.name)

## 2. Prompt Playground (UI) 🎮

Before you write any code, try a fix interactively:

1. In your project, open a trace where the agent **hallucinated** (e.g., student loans, Medicaid, homebuyer program).
2. Click the **LLM span** (the first one under the `agent` node, e.g. `ChatOpenAI`) → **Open in Playground**. The system prompt, messages, tools, and model load automatically.
3. Change the system prompt. Add rules like *"Only use information from tools. If no program matches, say so and refer to the DCS hotline."*
4. Click **Run** and compare the output with the original. Try a different model too.
5. **Run the playground against your dataset:** pick the `civicassist-prod-failures-...` dataset in Playground to replay all of those questions at once.
6. Optional: **Save the prompt** to Prompt Hub so it's versioned.

🤔 Did the fix work on the one example? Good. **Did it break anything else?** You can't tell from one example. That's what experiments are for.

## 3. A golden dataset with reference answers

[`data/golden_dataset.csv`](../data/golden_dataset.csv) has 17 questions written with "subject matter experts." Each has an **expected answer** and the **expected tools**, covering every category of request.

In [ ]:
import pandas as pd

golden = pd.read_csv("data/golden_dataset.csv").fillna("")
golden_ds = client.datasets.create(
    name=f"civicassist-golden-{handle}-{stamp}",
    space=SPACE_ID,
    examples=golden.to_dict(orient="records"),
)
DATASET = golden_ds.name
print("Created dataset:", DATASET)
golden.groupby("category").size()

## 4. Define the task and evaluators

**Task:** run the agent on each `question` with a given prompt version and model.

**Evaluators:**
- `correctness` (LLM judge): does the answer match the expected answer?
- `tool_call_accuracy` (code): were all `expected_tools` called? Here we have ground truth, so no judge is needed.
- `cites_source` (code): does the answer include the official link?

In [ ]:
from civic_agent import ask
from arize.experiments import EvaluationResult
from phoenix.evals import create_classifier
from workshop_utils import judge_model

def make_task(prompt_version: str, model: str | None = None):
    def task(dataset_row) -> dict:
        r = ask(dataset_row["question"], prompt_version=prompt_version, model=model,
                trace_to_project=False)  # keep experiment runs out of "production" traces
        return {"answer": r["answer"], "tools_called": r["tools_called"]}
    return task

judge = judge_model()
correctness_judge = create_classifier(
    name="correctness",
    llm=judge,
    choices={"correct": 1.0, "incorrect": 0.0},
    prompt_template="""Compare an AI assistant's answer to a reference answer written by a subject matter expert.

[Question]: {{question}}
[Reference answer]: {{expected_answer}}
[Assistant answer]: {{answer}}

Respond "correct" if the assistant's answer reaches the same conclusion and includes the key
facts from the reference (numbers, dates, requirements, next steps). Extra helpful detail is
fine; contradicting or missing key facts is not. Otherwise respond "incorrect".""",
)

def correctness(output, dataset_row) -> EvaluationResult:
    s = correctness_judge.evaluate({
        "question": dataset_row["question"],
        "expected_answer": dataset_row["expected_answer"],
        "answer": output["answer"],
    })[0]
    return EvaluationResult(score=s.score, label=s.label, explanation=s.explanation)

def tool_call_accuracy(output, dataset_row) -> EvaluationResult:
    expected = {t.strip() for t in str(dataset_row.get("expected_tools") or "").split(",") if t.strip()}
    called = set(output["tools_called"])
    missing = expected - called
    ok = not missing
    return EvaluationResult(
        score=1.0 if ok else 0.0, label="correct" if ok else "incorrect",
        explanation=f"expected={sorted(expected)} called={sorted(called)} missing={sorted(missing)}",
    )

def cites_source(output) -> EvaluationResult:
    ok = "dcs.columbia.example.gov" in output["answer"]
    return EvaluationResult(score=1.0 if ok else 0.0, label="cited" if ok else "not_cited")

EVALUATORS = [correctness, tool_call_accuracy, cites_source]

Do a quick **dry run** on 2 rows (nothing is uploaded) to catch bugs before spending tokens on the full run:

In [ ]:
_, dry = client.experiments.run(
    name="dry-run", dataset=DATASET, space=SPACE_ID,
    task=make_task("v1"), evaluators=EVALUATORS, dry_run=True, dry_run_count=2,
)
dry.filter(regex="output|score|label").head()

## 5. Experiment A: the baseline (prompt v1)
Takes about 2–3 minutes.

In [ ]:
exp_v1, df_v1 = client.experiments.run(
    name=f"v1-baseline-{stamp}", dataset=DATASET, space=SPACE_ID,
    task=make_task("v1"), evaluators=EVALUATORS, concurrency=4,
)

## 6. Experiment B: the improved prompt (v2)

Now look at the fixed prompt. Compare it with what you tried in Playground:

In [ ]:
from civic_agent.prompts import PROMPT_V2
print(PROMPT_V2)

In [ ]:
exp_v2, df_v2 = client.experiments.run(
    name=f"v2-grounded-prompt-{stamp}", dataset=DATASET, space=SPACE_ID,
    task=make_task("v2"), evaluators=EVALUATORS, concurrency=4,
)

## 7. Experiment C (optional): same prompt, cheaper model

Can a smaller model with the better prompt hold quality at a fraction of the cost? This question matters for the ROI discussion in Lab 5.

In [ ]:
from civic_agent import default_model

SMALL_MODEL = default_model("small")  # gpt-4.1-nano, claude-haiku-5-5, ... (set CIVIC_SMALL_MODEL to change)
print("Cheaper model:", SMALL_MODEL)
exp_small, df_small = client.experiments.run(
    name=f"v2-small-{stamp}", dataset=DATASET, space=SPACE_ID,
    task=make_task("v2", model=SMALL_MODEL), evaluators=EVALUATORS, concurrency=4,
)

## 8. Compare side by side

In [ ]:
def score_summary(df, label):
    cols = [c for c in df.columns if c.endswith(".score") or c.endswith("_score") or c.endswith(".score.value")]
    return df[cols].apply(pd.to_numeric, errors="coerce").mean().rename(label)

runs = {"v1 baseline": df_v1, "v2 prompt": df_v2}
if "df_small" in globals():
    runs["v2 + small model"] = df_small
comparison = pd.concat([score_summary(df, name) for name, df in runs.items()], axis=1)
comparison.style.format("{:.0%}").background_gradient(axis=1, cmap="RdYlGn", vmin=0, vmax=1)

### In the Arize UI 🔍

1. Go to **Datasets → `civicassist-golden-...` → Experiments**.
2. Select the experiments and click **Compare**. You'll see each question's output and eval label side by side across experiments.
3. Look for **regressions**: rows that v1 got right and v2 got wrong. A higher average can still hide a regression on a question that matters.
4. Click into a run to see its trace.

🤔 **Discuss:**
- Did v2 fix the hallucinations *and* the tool-use problems? What about the `missing_info` row?
- Did the cheaper model hold up? Where did it fail?
- What would you need to see before shipping v2 to production?

**Next:** [Lab 4: Signal & dashboards →](04_signal_and_dashboards.ipynb)